In [0]:
%sql
USE CATALOG V_Commerce;

CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType
from itertools import chain
import requests

In [0]:
def normalize_text(column):
    return F.lower(
        F.translate(
            F.trim(column),
            "áàãâäéèêëíìîïóòõôöúùûüçÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇ",
            "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC"
        )
    )

url_ibge = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

response = requests.get(url_ibge)
response.raise_for_status()

municipios = response.json()

rows = []

for municipio in municipios:
    nome_cidade = municipio.get("nome")

    microrregiao = municipio.get("microrregiao") or {}
    mesoregiao  = microrregiao.get("mesorregiao") or {}
    uf          = mesoregiao.get("UF") or {}

    sigla_uf = uf.get("sigla")

    if nome_cidade and sigla_uf:
        rows.append((nome_cidade, sigla_uf))

df_ibge = spark.createDataFrame(
    rows,
    ["cidade_ibge", "uf_ibge"]
)

df_ibge = (
    df_ibge
    .withColumn("cidade_norm", normalize_text(F.col("cidade_ibge")))
    .dropDuplicates(["cidade_norm"])
)

In [0]:
def verify_null_colluns(table):
    table.select([

        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in table.columns

    ]).display()

In [0]:
catalog = "V_Commerce"
silver_schema_name = "silver"

In [0]:
tb_avaliacoes_bronze = spark.table("v_commerce.bronze.tb_avaliacoes")
tb_catalogo_produtos_bronze = spark.table("v_commerce.bronze.tb_catalogo_produtos")
tb_clickstream_bronze = spark.table("v_commerce.bronze.tb_clickstream")
tb_clientes_bronze = spark.table("v_commerce.bronze.tb_clientes")
tb_pedidos_bronze = spark.table("v_commerce.bronze.tb_pedidos")
tb_suporte_tickets_bronze = spark.table("v_commerce.bronze.tb_suporte_tickets")

In [0]:
tb_pedidos_bronze.filter(
    F.expr("try_cast(quantidade AS INT)") < 0
).count()

8828

In [0]:
tb_pedidos_bronze.filter(
    F.expr("try_cast(valor_pedido AS FLOAT)") < 0
).count()

6246

In [0]:
# prata.tb_pedidos
# Origem: bronze.tb_pedidos

tb_pedidos_silver = (
    spark.table("v_commerce.bronze.tb_pedidos")

    .withColumn(
        "_rank",
        F.row_number().over(
            Window.partitionBy("id_pedido").orderBy(F.col("timestamp_ingestion_bronze").desc())
        )
    )
    .filter(F.col("_rank") == 1)
    .drop("_rank")

    .select(
        F.col("id_pedido"),
        F.col("id_cliente"),
        F.col("id_produto"),

        F.round(
            F.when(
                F.regexp_replace(
                    F.regexp_replace(F.col("valor_pedido"), r"R\$|\s+", ""),
                    r",", "."
                ).cast(DoubleType()) < 0,
                -1
            ).otherwise(
                F.regexp_replace(
                    F.regexp_replace(F.col("valor_pedido"), r"R\$|\s+", ""),
                    r",", "."
                ).cast(DoubleType())
            ), 2
        ).alias("valor_pedido"),

        F.coalesce(
            F.expr("try_to_date(data_pedido, 'yyyy-MM-dd')"),
            F.expr("try_to_date(data_pedido, 'dd-MM-yyyy')"),
            F.expr("try_to_date(data_pedido, 'MM-dd-yyyy')"),
            F.expr("try_to_date(data_pedido, 'dd/MM/yyyy')"),
            F.expr("try_to_date(data_pedido, 'MM/yyyy/dd')"),
            F.expr("try_to_date(data_pedido, 'yyyy/MM/dd')"),
            F.expr("try_to_date(data_pedido, 'yyyy/dd/MM')"),
            F.expr("try_to_date(data_pedido, 'dd-yyyy-MM')"),
            F.expr("try_to_date(data_pedido, 'MM-yyyy-dd')"),
        ).alias("data_pedido"),

        F.when(F.upper(F.col("metodo_pagamento")).rlike(r"^P[1I]X$"),                    "PIX")
         .when(F.upper(F.col("metodo_pagamento")).rlike(r"^B[0O]L(ET[0O])?$"),           "Boleto")
         .when(F.upper(F.col("metodo_pagamento")).rlike(r"^C[@A]RT([AÃ][OÃ])?$|^CRT$"), "Cartao")
         .otherwise(None)
         .alias("metodo_pagamento"),

        F.when(F.upper(F.col("status")).rlike(r"^APROV(ADO{1,2})?$|^APR$"),          "Aprovado")
         .when(F.upper(F.col("status")).rlike(r"^RECUS(ADO{1,2})?$|^REC$"),          "Recusado")
         .when(F.upper(F.col("status")).rlike(r"^PROC(ESS(ANDO)?)?$"),               "Processando")
         .when(F.upper(F.col("status")).rlike(r"^REEMB(OLS(O|AD[OA]?)?)?$"),         "Reembolsado")
         .otherwise(None)
         .alias("status"),

        F.when(F.col("quantidade") == "um",     1)
         .when(F.col("quantidade") == "dois",   2)
         .when(F.col("quantidade") == "tres",   3)
         .when(F.col("quantidade") == "quatro", 4)
         .when(F.col("quantidade") == "cinco",  5)
         .when(F.expr("try_cast(try_cast(quantidade AS FLOAT) AS INT)") <= 0, -1)
         .otherwise(F.expr("try_cast(try_cast(quantidade AS FLOAT) AS INT)"))
         .alias("quantidade"),

        F.current_timestamp().alias("timestamp_ingestion_silver"),
    )
)

tb_pedidos_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{silver_schema_name}.tb_pedidos")
print(f"✅ Tabela {catalog}.{silver_schema_name}.tb_pedidos criada com sucesso!\n")

✅ Tabela V_Commerce.silver.tb_pedidos criada com sucesso!



In [0]:
#tb_avaliacoes_bronze.select("data_avaliacao").distinct().show()


In [0]:
# prata.tb_avaliacoes
# Origem: bronze.tb_avaliacoes

tb_avaliacoes_silver = (
    spark.table("v_commerce.bronze.tb_avaliacoes")

    .withColumn(
        "_rank",
        F.row_number().over(
            Window.partitionBy("id_avaliacao").orderBy(F.col("timestamp_ingestion_bronze").desc())
        )
    )
    .filter(F.col("_rank") == 1)
    .drop("_rank")

    .select(
        F.col("id_avaliacao"),
        F.col("id_pedido"),
        F.col("id_cliente"),
        F.col("id_produto"),

        F.when(F.col("nota_produto") == "péssimo", 1)
         .when(F.col("nota_produto") == "ruim",    2)
         .when(F.col("nota_produto") == "bom",     4)
         .when(F.col("nota_produto") == "ótimo",   5)
         .when(F.expr("try_cast(nota_produto AS INT)").between(1, 5),
               F.expr("try_cast(nota_produto AS INT)"))
         .otherwise(-1)
         .alias("nota_produto"),

        F.col("comentario"),

        F.when(F.col("nota_nps") == "péssimo", 1)
         .when(F.col("nota_nps") == "ruim",    3)
         .when(F.col("nota_nps") == "bom",     8)
         .when(F.col("nota_nps") == "ótimo",   10)
         .when(F.expr("try_cast(nota_nps AS INT)").between(0, 10),
               F.expr("try_cast(nota_nps AS INT)"))
         .otherwise(-1)
         .alias("nota_nps"),

        F.when(F.lower(F.col("recomenda")).isin("s", "sim", "yes", "1"), "Sim")
         .when(F.lower(F.col("recomenda")).isin("n", "nao", "não", "no", "0"), "Não")
         .otherwise(None)
         .alias("recomenda"),

        F.coalesce(
            # com timestamp
            F.expr("try_to_date(data_avaliacao, 'yyyy-MM-dd HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'yyyy/dd/MM HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'yyyy/MM/dd HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'dd-MM-yyyy HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'MM-dd-yyyy HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'dd/MM/yyyy HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'MM/yyyy/dd HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'dd-yyyy-MM HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'MM-yyyy-dd HH:mm:ss')"),
            # sem timestamp
            F.expr("try_to_date(data_avaliacao, 'yyyy-MM-dd')"),
            F.expr("try_to_date(data_avaliacao, 'yyyy/dd/MM')"),
            F.expr("try_to_date(data_avaliacao, 'yyyy/MM/dd')"),
            F.expr("try_to_date(data_avaliacao, 'dd-MM-yyyy')"),
            F.expr("try_to_date(data_avaliacao, 'MM-dd-yyyy')"),
            F.expr("try_to_date(data_avaliacao, 'dd/MM/yyyy')"),
            F.expr("try_to_date(data_avaliacao, 'MM/yyyy/dd')"),
            F.expr("try_to_date(data_avaliacao, 'dd-yyyy-MM')"),
            F.expr("try_to_date(data_avaliacao, 'MM-yyyy-dd')"),
        ).alias("data_avaliacao"),

        F.current_timestamp().alias("timestamp_ingestion_silver"),
    )
)

tb_avaliacoes_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{silver_schema_name}.tb_avaliacoes")
print(f"✅ Tabela {catalog}.{silver_schema_name}.tb_avaliacoes criada com sucesso!\n")

✅ Tabela V_Commerce.silver.tb_avaliacoes criada com sucesso!



In [0]:
# prata.tb_clickstream
# Origem: bronze.tb_clickstream

column_dispositivo_treated = F.trim(F.lower(F.col("dispositivo")))
column_evento_treated = F.trim(F.lower(F.col("tipo_evento")))
column_canal_treated = F.regexp_replace(F.trim(F.lower(F.col("canal"))), r"\s+", "_")

tb_clickstream_silver = (
    tb_clickstream_bronze

    .withColumn(
        "_rank",
        F.row_number().over(
            Window.partitionBy("id_evento").orderBy(F.col("timestamp_ingestion_bronze").desc())
        )
    )
    .filter(F.col("_rank") == 1)
    .drop("_rank")

    .select(
        F.col("id_evento"),
        F.col("id_sessao"),
        F.coalesce(F.col("id_cliente"), F.lit("usuario anônimo")).alias("id_cliente"),
        F.col("id_dispositivo"),
        F.coalesce(F.col("id_produto"), F.lit("N/A")).alias("id_produto"),

        F.when(column_evento_treated.isin("pageview", "page-view"), F.lit("page_view"))
         .when(column_evento_treated.isin("busca", "srch"), F.lit("search"))
         .when(column_evento_treated.isin("adicionar", "add-to-cart", "addtocart"), F.lit("add_to_cart"))
         .when(column_evento_treated.isin("prod-view", "product-view", "productview", "prod_view"), F.lit("product_view"))
         .when(column_evento_treated.isin("abandon-cart", "abandoncart", "abandono", ), F.lit("abandon_cart"))
         .when(column_evento_treated.isin("login", "singin", "signin", "log-in", "sing-in"), F.lit("log_in"))
         .when(column_evento_treated.isin("compra", "buy"), F.lit("purchase"))
         .when(column_evento_treated.isin("pagamento"), F.lit("payment"))
         .when(column_evento_treated.isin("pv"), F.lit("no_specified"))
         .when(column_evento_treated.isin("check_out"), F.lit("checkout"))
         .otherwise(column_evento_treated).alias("tipo_evento"),

        F.when(column_canal_treated.isin("aplicativo"), F.lit("app"))
         .otherwise(column_canal_treated).alias("canal"),

        F.when(column_dispositivo_treated.isin("computador"), F.lit("desktop"))
         .when(column_dispositivo_treated.isin("mob", "celular"), F.lit("mobile"))
         .when(column_dispositivo_treated.isin("tab"), F.lit("tablet"))
         .otherwise(column_dispositivo_treated).alias("dispositivo"),

        F.col("origem_sessao"),
        F.col("data_evento"),
        F.coalesce(F.col("tempo_pagina_seg"), F.lit(0)).alias("tempo_pagina_seg"),
        F.current_timestamp().alias("timestamp_ingestion_silver")
    )
)

tb_clickstream_silver.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{catalog}.{silver_schema_name}.tb_clickstream")
print(f"✅ Tabela {catalog}.{silver_schema_name}.tb_clickstream criada com sucesso!\n")

✅ Tabela V_Commerce.silver.tb_clickstream criada com sucesso!



In [0]:
# prata.tb_suporte_tickets
# Origem: bronze.tb_suporte_tickets

column_tipo_problema_treated = F.trim(F.lower(F.col("tipo_problema")))
column_agente_suporte_treated = F.initcap(F.trim(F.lower(F.col("agente_suporte"))))

tb_suporte_tickets_silver = (
    tb_suporte_tickets_bronze

    .withColumn(
        "_rank",
        F.row_number().over(
            Window.partitionBy("ticket_id").orderBy(F.col("timestamp_ingestion_bronze").desc())
        )
    )
    .filter(F.col("_rank") == 1)
    .drop("_rank")

    .select(
        F.col("ticket_id").alias('id_ticket'),
        F.col("id_cliente"),
        F.col("id_pedido"),
        
        # 1. Tratamento do tipo_problema (agrupando pt, en e erros de digitação)
        F.when(column_tipo_problema_treated.isin("pro", "produto", "p3oduto", "product", "prod"), F.lit("product"))
         .when(column_tipo_problema_treated.isin("pagamento", "p4gamento", "pag", "payment", "pay"), F.lit("payment"))
         .when(column_tipo_problema_treated.isin("entrega", "3ntrega", "entr"), F.lit("delivery"))
         .when(column_tipo_problema_treated.isin("ref", "reembolso", "r3embolso", "reemb", "refund"), F.lit("refund"))
         .when(column_tipo_problema_treated.isin("del", "delay", ), F.lit("delay"))
         .otherwise(column_tipo_problema_treated).alias("tipo_problema"),
        
        F.col("data_abertura"),
        
        # Nulos mantidos para representar que os tickets não foram resolvidos ainda
        F.col("data_resolucao"),
        F.col("tempo_resolucao_horas"),
        
        column_agente_suporte_treated.alias("agente_suporte"),
        
        # Os nulos serão substituídos pelo valor -1 para não alterar a nota ao mesmo tempo que não fica como nulo
        F.coalesce(F.col("nota_avaliacao"), F.lit(-1)).alias("nota_avaliacao"),
        
        F.current_timestamp().alias("timestamp_ingestion_silver")
    )
)

tb_suporte_tickets_silver.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{catalog}.{silver_schema_name}.tb_suporte_tickets")
print(f"✅ Tabela {catalog}.{silver_schema_name}.tb_suporte_tickets criada com sucesso!\n")

✅ Tabela V_Commerce.silver.tb_suporte_tickets criada com sucesso!



In [0]:
# silver.tb_clientes
# Origem: bronze.tb_clientes

ufs_validas = [
    "AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS",
    "MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC",
    "SP","SE","TO"
]

mapa_estados = {
    "acre": "AC", "alagoas": "AL", "amapa": "AP", "amazonas": "AM",
    "bahia": "BA", "ceara": "CE", "distrito federal": "DF",
    "espirito santo": "ES", "goias": "GO", "maranhao": "MA",
    "mato grosso":
 "MT", "mato grosso do sul": "MS",
    "minas gerais": "MG", "para": "PA", "paraiba": "PB",
    "parana": "PR", "pernambuco": "PE", "piaui": "PI",
    "rio de janeiro": "RJ", "rio grande do norte": "RN",
    "rio grande do sul": "RS", "rondonia": "RO", "roraima": "RR",
    "santa catarina": "SC", "sao paulo": "SP",
    "sergipe": "SE", "tocantins": "TO"
}

mapping_estados = F.create_map([F.lit(x) for x in chain(*mapa_estados.items())])  # Mapeamento de nomes para siglas

email_limpo = F.lower(F.trim(F.col("email")))

email_corrigido = F.when(
    email_limpo.rlike(
        r"^[a-z0-9._%+-]+(gmail\.com|hotmail\.com|outlook\.com|yahoo\.com|email\.com|uol\.com\.br|bol\.com\.br|icloud\.com)$"
    ),
    F.regexp_replace(
        email_limpo,
        r"(gmail\.com|hotmail\.com|outlook\.com|yahoo\.com|email\.com|uol\.com\.br|bol\.com\.br|icloud\.com)$",
        r"@\1"
    )
).otherwise(email_limpo)

email_tratado = (
    F.when(
        F.col("email").isNull() |
        (F.trim(F.col("email")) == ""),
        F.lit("Email inválido")
    )

    # valida SOMENTE depois da correção
    .when(
        email_corrigido.rlike(
            r"^[a-z0-9._%+-]+@[a-z0-9-]+(\.[a-z]{2,})+$"
        ),
        email_corrigido
    )

    .otherwise(F.lit("Email inválido"))
)
tb_clientes_silver = (
    spark.table("v_commerce.bronze.tb_clientes")  # Carrega tabela bronze

    .withColumn(
        "_rank",
        F.row_number().over(
            Window.partitionBy("id_cliente").orderBy(
                F.to_date(F.col("data_cadastro")).desc_nulls_last()
            )
        )
    )
    .filter(F.col("_rank") == 1)  # Mantém apenas o registro mais recente
    .drop("_rank")

    # Cidade e estado
    .withColumn("cidade_original", F.trim(F.col("cidade")))
    .withColumn(
        "uf_extraida_cidade",
        F.upper(
            F.regexp_extract(
                F.col("cidade_original"),
                r"(?i)(?:^|[\s\-/])([A-Z]{2})(?:$|[\s\-/])",
                1
            )
        )
    )
    .withColumn(
        "cidade_limpa",
        F.regexp_replace(
            F.col("cidade_original"),
            r"(?i)[\s\-/]+[A-Z]{2}$",
            ""
        )
    )
    .withColumn("cidade_norm", normalize_text(F.col("cidade_limpa")))  # Normaliza cidade
    .withColumn("uf_estado_nome", mapping_estados[F.col("cidade_norm")])  # Busca sigla do estado

    # Telefone BR
    .withColumn(
        "ramal_extraido",
        F.regexp_extract(
            F.lower(F.col("telefone").cast("string")),
            r"(?:ramal|r\.?|ext\.?|x)\s*(\d+)",
            1
        )
    )
    .withColumn(
        "ramal",
        F.when(F.col("ramal_extraido") == "", F.lit("Não informado"))
        .otherwise(F.col("ramal_extraido"))
    )
    .withColumn(
        "telefone_sem_ramal",
        F.regexp_replace(
            F.lower(F.col("telefone").cast("string")),
            r"(?:ramal|r\.?|ext\.?|x)\s*\d+",
            ""
        )
    )
    .withColumn(
        "telefone_limpo",
        F.regexp_replace(F.col("telefone_sem_ramal"), r"[^0-9]", "")
    )
    .withColumn(
        "telefone_sem_55",
        F.when(
            F.col("telefone_limpo").startswith("55") & (F.length(F.col("telefone_limpo")) > 11),
            F.expr("substring(telefone_limpo, 3, length(telefone_limpo))")
        ).otherwise(F.col("telefone_limpo"))
    )
    .withColumn(
        "telefone_tratado",
        F.when(
            F.col("telefone_sem_55").rlike(r"^\d{10,11}$"),
            F.col("telefone_sem_55")
        ).otherwise(F.lit("Não informado"))
    )

    # Endereço padrão BR: logradouro, número e complemento
    .withColumn("endereco_original", F.col("endereco").cast("string"))
    .withColumn(
        "complemento_extraido",
        F.regexp_extract(
            F.lower(F.col("endereco_original")),
            r"(apt\.?\s*\d+|apto\.?\s*\d+|suite\s*\d+|sala\s*\d+|bloco\s*\w+)",
            1
        )
    )
    .withColumn(
        "complemento",
        F.when(F.col("complemento_extraido") == "", F.lit("Não informado"))
        .otherwise(F.initcap(F.col("complemento_extraido")))
    )
    .withColumn(
        "numero_endereco",
        F.regexp_extract(F.col("endereco_original"), r"(\d+)", 1)
    )
    .withColumn(
        "numero_endereco",
        F.when(F.col("numero_endereco") == "", F.lit("Não informado"))
        .otherwise(F.col("numero_endereco"))
    )
    .withColumn(
        "endereco_sem_complemento",
        F.regexp_replace(
            F.lower(F.col("endereco_original")),
            r"(apt\.?\s*\d+|apto\.?\s*\d+|suite\s*\d+|sala\s*\d+|bloco\s*\w+)",
            ""
        )
    )
    .withColumn(
        "logradouro",
        F.regexp_replace(F.col("endereco_sem_complemento"), r"\d+", "")
    )
    .withColumn(
        "logradouro",
        F.initcap(F.trim(F.regexp_replace(F.col("logradouro"), r"\s+", " ")))
    )
    .withColumn(
        "endereco_tratado",
        F.when(
            (F.col("logradouro") == "") |
            (F.col("numero_endereco") == "Não informado") |
            F.col("endereco_original").isNull(),
            F.lit("Não informado")
        ).otherwise(
            F.concat_ws(", ", F.col("logradouro"), F.col("numero_endereco"))
        )
    )
    .withColumn(
        "endereco_valido",
        F.when(F.col("endereco_tratado") != "Não informado", F.lit("Sim"))
        .otherwise(F.lit("Não"))
    )

    .join(
        df_ibge.select("cidade_norm", "uf_ibge"),
        on="cidade_norm",
        how="left"
    )  # Enriquecimento com dados IBGE

    .select(
        F.col("id_cliente"),

        F.initcap(F.trim(F.col("nome"))).alias("nome"),
        F.initcap(F.trim(F.col("sobrenome"))).alias("sobrenome"),

        email_tratado.alias("email"),

        F.col("telefone_tratado").alias("telefone"),
        F.col("ramal"),
        F.when(
            F.col("telefone_tratado").rlike(r"^\d{10,11}$"),
            F.lit("Sim")
        ).otherwise(F.lit("Não")).alias("telefone_valido"),

        F.col("endereco_tratado").alias("endereco"),
        F.col("numero_endereco"),
        F.col("complemento"),
        F.col("endereco_valido"),

        F.to_date(F.col("data_nascimento")).alias("data_nascimento"),
        F.to_date(F.col("data_cadastro")).alias("data_cadastro"),

        F.when(
            F.upper(F.trim(F.col("genero"))).isin("M", "F"),
            F.upper(F.trim(F.col("genero")))
        ).otherwise(F.lit("Não informado")).alias("genero"),

        F.coalesce(
            F.initcap(F.trim(F.col("cidade_limpa"))),
            F.lit("Não informado")
        ).alias("cidade"),

        F.when(F.col("uf_ibge").isNotNull(), F.col("uf_ibge"))
        .when(F.col("uf_estado_nome").isNotNull(), F.col("uf_estado_nome"))
        .when(F.col("uf_extraida_cidade").isin(ufs_validas), F.col("uf_extraida_cidade"))
        .when(F.upper(F.trim(F.col("estado"))).isin(ufs_validas), F.upper(F.trim(F.col("estado"))))
        .otherwise(F.lit("Não informado"))
        .alias("estado"),  # Normalização do estado

        F.when(normalize_text(F.col("origem")) == "indicacao", F.lit("Indicação"))
        .when(normalize_text(F.col("origem")) == "web", F.lit("Web"))
        .when(normalize_text(F.col("origem")) == "app", F.lit("App"))
        .otherwise(F.coalesce(F.initcap(F.trim(F.col("origem"))), F.lit("Não informado")))
        .alias("origem"),

        F.when(
            F.col("device_ids").isNull() | (F.trim(F.col("device_ids")) == ""),
            F.lit("Não informado")
        ).otherwise(F.col("device_ids")).alias("ids_dispositivos"),

        F.year(F.to_date(F.col("data_cadastro"))).alias("ano_cadastro"),

        F.floor(
            F.datediff(F.current_date(), F.to_date(F.col("data_nascimento"))) / 365.25
        ).alias("idade_aproximada"),

        F.when(
            (email_tratado == "Email inválido") |
            (F.col("telefone_tratado") == "Não informado") |
            (F.col("endereco_tratado") == "Não informado") |
            F.col("data_cadastro").isNull() |
            F.col("id_cliente").isNull(),
            F.lit("Sim")
        ).otherwise(F.lit("Não")).alias("precisa_revisao"),  # Validação de dados

        F.current_timestamp().alias("timestamp_ingestion_silver")
    )
)

tb_clientes_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{silver_schema_name}.tb_clientes")  # Salva tabela silver

print(f"✅ Tabela {catalog}.{silver_schema_name}.tb_clientes criada com sucesso!\n")

✅ Tabela V_Commerce.silver.tb_clientes criada com sucesso!



In [0]:
# silver.tb_catalogo_produtos
# Origem: bronze.tb_catalogo_produtos

tb_catalogo_produtos_silver = (
    spark.table("v_commerce.bronze.tb_catalogo_produtos")  

    .withColumn(
        "_rank",
        F.row_number().over(
            Window.partitionBy("id_produto").orderBy(
                F.to_date(F.col("data_cadastro_produto")).desc_nulls_last()
            )
        )
    )  # Gera ranking para pegar o registro mais recente por produto
    .filter(F.col("_rank") == 1)  # Filtra apenas o registro mais recente
    .drop("_rank")  # Remove coluna auxiliar

    .select(
        F.col("id_produto"),

        F.initcap(F.trim(F.col("nome_produto"))).alias("nome_produto"),  # Normaliza nome
        F.initcap(F.trim(F.col("fornecedor"))).alias("fornecedor"),  # Normaliza fornecedor

        F.to_date(F.col("data_cadastro_produto")).alias("data_cadastro_produto"),  # Converte data

        # Normaliza preço, substitui valores inválidos por -1
        F.when(
            F.expr("try_cast(regexp_replace(regexp_replace(cast(preco AS string), '[R$\\s]', ''), ',', '.') AS decimal(10,2))").isNull() |
            (F.expr("try_cast(regexp_replace(regexp_replace(cast(preco AS string), '[R$\\s]', ''), ',', '.') AS decimal(10,2))") <= 0),
            F.lit(-1).cast("decimal(10,2)")
        ).otherwise(
            F.expr("try_cast(regexp_replace(regexp_replace(cast(preco AS string), '[R$\\s]', ''), ',', '.') AS decimal(10,2))")
        ).alias("preco"),

        # Classifica categoria com base em palavras-chave
        F.when(
            F.col("categoria").isNull(),
            "Não informado"
        )
        .when(F.lower(F.col("categoria")).contains("mov"), "Móveis")
        .when(F.lower(F.col("categoria")).contains("brin"), "Brinquedos")
        .when(F.lower(F.col("categoria")).contains("aut"), "Automotivo")
        .when(F.lower(F.col("categoria")).contains("elet"), "Eletrônicos")
        .when(F.lower(F.col("categoria")).contains("vest"), "Vestuário")
        .when(F.lower(F.col("categoria")).contains("cas"), "Casa")
        .when(F.lower(F.col("categoria")).contains("bel"), "Beleza")
        .when(F.lower(F.col("categoria")).contains("esp"), "Esportes")
        .otherwise("Outros")
        .alias("categoria"),

        # Normaliza status de ativo
        F.when(
            F.col("ativo").isNull() |
            F.lower(F.trim(F.col("ativo").cast("string"))).isin("", "null", "none"),
            "Não identificado"
        )
        .when(
            F.lower(F.trim(F.col("ativo").cast("string"))).isin("sim", "s", "1", "true"),
            "Sim"
        )
        .when(
            F.lower(F.trim(F.col("ativo").cast("string"))).isin("nao", "não", "n", "0", "false"),
            "Não"
        )
        .otherwise("Não identificado")
        .alias("ativo"),

        F.when(
            F.expr("try_cast(avaliacao_interna AS double)") < 0,
            F.lit(-1.0)
        ).when(
            F.expr("try_cast(avaliacao_interna AS double)") > 5,
            F.lit(-1.0)
        ).otherwise(
            F.expr("try_cast(avaliacao_interna AS double)")
        ).alias("avaliacao_interna"),

        # Normaliza peso, substitui valores inválidos por -1
        F.when(
            F.expr("try_cast(peso_kg AS double)").isNull() |
            (F.expr("try_cast(peso_kg AS double)") <= 0),
            F.lit(-1.0)
        ).otherwise(F.expr("try_cast(peso_kg AS double)"))
        .alias("peso_kg"),

        # --- AJUSTE: Normaliza estoque, valores nulos ou inválidos viram -1 ---
        F.coalesce(
            F.expr("try_cast(estoque_disponivel AS int)"),
            F.lit(-1)
        ).alias("estoque_disponivel"),

        # Marca se tem estoque disponível (considerando apenas valores > 0)
        F.when(
            F.coalesce(F.expr("try_cast(estoque_disponivel AS int)"), F.lit(-1)) > 0,
            "Sim"
        ).otherwise("Não").alias("tem_estoque"),

        # Marca se precisa revisão (peso, preço ou estoque inválido)
        F.when(
            F.expr("try_cast(peso_kg AS double)").isNull() |
            (F.expr("try_cast(peso_kg AS double)") <= 0) |
            F.expr("try_cast(regexp_replace(regexp_replace(cast(preco AS string), '[R$\\s]', ''), ',', '.') AS decimal(10,2))").isNull() |
            F.expr("try_cast(estoque_disponivel AS int)").isNull() |
            (F.expr("try_cast(avaliacao_interna AS double)") < 0) |
            (F.expr("try_cast(avaliacao_interna AS double)") > 5),
            "Sim"
        ).otherwise("Não").alias("precisa_revisao"),

        F.current_timestamp().alias("timestamp_ingestion_silver")  # Converte timestamp
    )
)

tb_catalogo_produtos_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{silver_schema_name}.tb_catalogo_produtos")  # Salva tabela silver

print(f"✅ Tabela {catalog}.{silver_schema_name}.tb_catalogo_produtos criada com sucesso!\n")

✅ Tabela silver.tb_catalogo_produtos criada com sucesso!

